# Swarm Clock, Part 3b: schema discovery for the replication corpus (zenml/mantiq-swarm-traces, CC-BY-4.0)
A cooperative 18-agent engineering swarm with TRUE agent ids and OS-level timestamps. Purpose: test, with ground-truth identity, whether cross-agent artifact similarity carries chronology in a swarm that genuinely builds on shared work.
Can run in a fresh runtime. Downloads only small tables first.

In [ ]:
!pip -q install huggingface_hub pyarrow
from huggingface_hub import HfApi, hf_hub_download
import pandas as pd, numpy as np, collections, json
REPO = "zenml/mantiq-swarm-traces"
api = HfApi()
tree = [f for f in api.list_repo_tree(REPO, repo_type="dataset", recursive=True) if hasattr(f, "size")]
T = pd.DataFrame([{"path": f.path, "size_mb": round((f.size or 0)/1e6, 2)} for f in tree])
T["config"] = T.path.str.split("/").str[0]
print(T.groupby("config").agg(files=("path","count"), total_mb=("size_mb","sum")).sort_values("total_mb").to_string())
print([p for p in T.path if p.endswith(".md")])

In [ ]:
for doc in [p for p in T.path if p.lower().endswith(("readme.md","schema.md"))][:2]:
    txt = open(hf_hub_download(REPO, doc, repo_type="dataset")).read()
    print("=====", doc, len(txt)); print(txt[:6000])

In [ ]:
practice = sorted({p.split("/")[-1] for p in T.path if "/practice/" in p and p.endswith(".parquet")})
print("practice runs:", practice)
def load(config, run, split="practice"):
    p = f"{config}/{split}/{run}"
    if p not in set(T.path): return None
    return pd.read_parquet(hf_hub_download(REPO, p, repo_type="dataset"))
RUN = practice[0] if practice else None
small = {}
for cfg in ["runs","agents","sessions","messages","blobs","model_calls","tool_calls","file_versions"]:
    df = load(cfg, RUN)
    if df is None: print(cfg, "missing for", RUN); continue
    small[cfg] = df
    print(f"\n=== {cfg} {df.shape}"); print(df.dtypes.to_string()); print(df.head(3).to_string()[:1500])

In [ ]:
# How is text reachable? Look for sha256 / blob references in each table
if "blobs" in small:
    b = small["blobs"]; print("blobs roles:", collections.Counter(map(str, b.get("roles", pd.Series(dtype=str)))).most_common(10))
    print("blob text length quantiles:", b.text.str.len().quantile([.5,.9,.99]).tolist() if "text" in b else "no text col")
for name, df in small.items():
    cols = [c for c in df.columns if any(k in c.lower() for k in ["sha","blob","text","content","payload","path","ts","time","agent"])]
    print(name, cols)
print("events file size (MB) for", RUN, T[T.path == f"events/practice/{RUN}"].size_mb.tolist())